# Bias generator v4: bias vs rule, with the extended event-calculus templates (Kaggle)
Settings: **Accelerator → GPU T4 x2**, **Internet → On**, then **Save Version → Save & Run All (Commit)**.

Two models trained on the same programs (old families + v3 families + the new v4 event-calculus families):
- **bias**: writes the mode declarations; XHAIL learns the rule (write + repair data, as in v3)
- **rule**: writes the rule directly

Both get up to 4 resampled attempts, 3 seeds, no retrieved examples. Takes about 4–5 hours.
Download **results_v4.zip** from the Output tab when it finishes.

In [ ]:
import glob, os, shutil
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
# pick the v4 dataset even if older xhail-bias datasets are also attached
src = next(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/pyproject.toml', recursive=True)
           if os.path.exists(os.path.join(os.path.dirname(p), 'experiments/bias_llm/data/sft_train_rule_v4.jsonl')))
shutil.copytree(src, '/kaggle/working/xhail', dirs_exist_ok=True)
%cd /kaggle/working/xhail
!pip install -q unsloth
!pip install -q -e .
%cd experiments/bias_llm
!wc -l data/sft_train_v4.jsonl data/sft_train_rule_v4.jsonl data/test.jsonl data/test_new.jsonl data/test_v4.jsonl

## 1. Train both models (bias ≈ 90 min, rule ≈ 40 min)

In [ ]:
if not os.path.exists('checkpoints/qwen-bias-v4/adapter_config.json'):
    !python train_unsloth.py --sft --sft-tag _v4 --model unsloth/Qwen2.5-1.5B-Instruct --epochs 2 --out checkpoints/qwen-bias-v4
if not os.path.exists('checkpoints/qwen-rule-v4/adapter_config.json'):
    !python train_unsloth.py --sft --sft-tag _rule_v4 --model unsloth/Qwen2.5-1.5B-Instruct --epochs 2 --out checkpoints/qwen-rule-v4

## 2. Generate: 2 jobs × 4 tiers × 3 seeds, up to 4 attempts (attempt 1 = single)
Tiers: **test** (160, old families), **new** (80, v3 families), **ec** (120, v4 event-calculus families), **gold** (16).
Then a sanity floor on 80 training programs per job.

In [ ]:
tiers = {'test': 'data/test.jsonl', 'new': 'data/test_new.jsonl', 'ec': 'data/test_v4.jsonl', 'gold': 'gold'}
models = {'bias': 'checkpoints/qwen-bias-v4', 'rule': 'checkpoints/qwen-rule-v4'}
for job, model in models.items():
    for tier, path in tiers.items():
        for seed in [0, 1, 2]:
            out = f'preds/v4/{job}_{tier}_resample_s{seed}.jsonl'
            if not os.path.exists(out):
                !python colab_eval.py --job {job} --model {model} --tasks {path} --strategy resample --seed {seed} --out {out}
    out = f'preds/v4/{job}_seen_resample_s0.jsonl'
    if not os.path.exists(out):
        !python colab_eval.py --job {job} --model {model} --tasks data/train_v4.jsonl --limit 80 --strategy resample --seed 0 --out {out}

## 3. Bundle results for download

In [ ]:
!cd /kaggle/working/xhail/experiments/bias_llm && zip -qr /kaggle/working/results_v4.zip preds/v4 checkpoints/qwen-bias-v4 checkpoints/qwen-rule-v4
!ls -la /kaggle/working/results_v4.zip